# Product Segmentation Case Study

## Context

When you think of sneakers for a trip, the importance of good footwear cannot be discarded, and the obvious brands that come to mind are Adidas and Nike. Adidas vs Nike is a constant debate as the two giants in the apparel market, with a large market cap and market share, battle it out to come on top. As a newly hired Data Scientist in a market research company, you have been given the task of extracting insights from the data of men's and women's shoes, and grouping products together to identify similarities and differences between the product range of these renowned brands.

## Let's start coding!

### Importing necessary libraries

In [ ]:




import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()


pd.set_option("display.max_columns", None)

pd.set_option("display.max_rows", 200)


from sklearn.preprocessing import StandardScaler


from scipy.spatial.distance import cdist


from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score




In [ ]:

data = pd.read_excel("Mall_Customers dataset.xlsx")

In [ ]:
data.shape

In [ ]:

data.sample(n=10, random_state=1)

In [ ]:

df = data.copy()

In [ ]:

df.columns = [c.replace(" ", "_") for c in df.columns]

In [ ]:
# let's look at the structure of the data
df.info()

**We won't need Product_ID for analysis, so let's drop this column.**

In [ ]:
df.drop("CustomerID", axis=1, inplace=True)

In [ ]:

df.duplicated().sum()

In [ ]:
df = df[(~df.duplicated())].copy()

**Let's take a look at the summary of the data**

In [ ]:
df.describe()

In [ ]:

(df['Annual_Income_(k$)'] == 0).sum()

In [ ]:

df[(df["Annual_Income_(k$)"] == 0)]

In [ ]:
df[(df['Annual_Income_(k$)'] == 0)].describe()

In [ ]:
df.loc[(df["Annual_Income_(k$)"] == 0), ["Annual_Income_(k$)"]] = df.loc[
    (df["Annual_Income_(k$)"] == 0), ["Age"]
].values

In [ ]:
df['Annual_Income_(k$)'].describe()

In [ ]:

df.isna().sum()

- There are no missing values in the data.

## EDA

In [ ]:



def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to the show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,
        sharex=True,
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette="winter"
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )

In [ ]:

num_col = df.select_dtypes(include=np.number).columns.tolist()

for item in num_col:
    histogram_boxplot(df, item)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(20, 15))
fig.suptitle("CDF plot of numerical variables", fontsize=20)
axes = axes.flatten()


for i, col in enumerate(num_col):
    if i < len(axes):
        sns.ecdfplot(ax=axes[i], x=df[col])


for j in range(len(num_col), len(axes)):
    fig.delaxes(axes[j])

fig.tight_layout(pad=2.0)
plt.show()

In [ ]:



def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n].sort_values(),
    )

    for p in ax.patches:
        if perc:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )
        else:
            label = p.get_height()

        x = p.get_x() + p.get_width() / 2
        y = p.get_height()

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )

    plt.show()

In [ ]:

labeled_barplot(df, "Annual_Income_(k$)", perc=True)

## Bivariate Analysis

**Let's check for correlations.**

In [ ]:
plt.figure(figsize=(15, 7))
sns.heatmap(df[num_col].corr(), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral")
plt.show()

In [ ]:
sns.pairplot(data=df[num_col], diag_kind="kde")
plt.show()

## Data Preprocessing

In [ ]:

num_col

In [ ]:

scaler = StandardScaler()
subset = df[num_col].copy()
subset_scaled = scaler.fit_transform(subset)

In [ ]:

subset_scaled_df = pd.DataFrame(subset_scaled, columns=subset.columns)

## K-means Clustering

In [ ]:
clusters = range(1, 9)
meanDistortions = []

for k in clusters:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(subset_scaled_df)
    prediction = model.predict(subset_scaled_df)
    distortion = (
        sum(
            np.min(cdist(subset_scaled_df, model.cluster_centers_, "euclidean"), axis=1)
        )
        / subset_scaled_df.shape[0]
    )

    meanDistortions.append(distortion)

    print("Number of Clusters:", k, "\tAverage Distortion:", distortion)

plt.plot(clusters, meanDistortions, "bx-")
plt.xlabel("k")
plt.ylabel("Average distortion")
plt.title("Selecting k with the Elbow Method")
plt.show()

**Let's check the silhouette scores.**

In [ ]:
sil_score = []
cluster_list = list(range(2, 10))
for n_clusters in cluster_list:
    clusterer = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    preds = clusterer.fit_predict((subset_scaled_df))

    score = silhouette_score(subset_scaled_df, preds)
    sil_score.append(score)
    print("For n_clusters = {}, the silhouette score is {})".format(n_clusters, score))

plt.plot(cluster_list, sil_score)
plt.show()

In [ ]:

kmeans = KMeans(n_clusters=4, random_state=0, n_init=1)
kmeans.fit(subset_scaled_df)

In [ ]:


df["K_means_segments"] = kmeans.labels_
subset_scaled_df["K_means_segments"] = kmeans.labels_

## Cluster Profiling

In [ ]:


df["K_means_segments"] = kmeans.labels_
subset_scaled_df["K_means_segments"] = kmeans.labels_


cluster_profile = df.groupby("K_means_segments")[num_col].mean()

cluster_profile["count_in_each_segments"] = (
    df.groupby("K_means_segments")["Annual_Income_(k$)"].count().values
)


cluster_profile.style.highlight_max(color="lightgreen", axis=0)

In [ ]:
cluster_profile["count_in_each_segments"] = (
    df.groupby("K_means_segments")["Annual_Income_(k$)"].count().values
)

In [ ]:

cluster_profile.style.highlight_max(color="lightgreen", axis=0)

In [ ]:

fig, axes = plt.subplots(1, len(num_col), figsize=(16, 6))
fig.suptitle("Boxplot of scaled numerical variables for each cluster", fontsize=20)

for ii, col in enumerate(num_col):
    sns.boxplot(
        ax=axes[ii],
        y=subset_scaled_df[col],
        x=subset_scaled_df["K_means_segments"],
    )


fig.tight_layout(pad=2.0)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(num_col), figsize=(16, 6))
fig.suptitle("Boxplot of original numerical variables for each cluster", fontsize=20)


for ii, col in enumerate(num_col):
    sns.boxplot(ax=axes[ii], y=df[col], x=df["K_means_segments"])

fig.tight_layout(pad=2.0)
plt.show()

In [ ]:
# pd.crosstab(df.K_means_segments, df.Brand).style.highlight_max(
#     color="lightgreen", axis=0
# )